# 5장 1강. LLM 워크플로우의 구성과 활용
## 실습문제


## 실습 목표

- 단일 호출, 순차 처리, 조건부 분기의 특징과 선택 이유를 설명할 수 있다.
- 앞 단계의 분류 결과와 원문을 다음 분석의 입력으로 전달할 수 있다.
- 미리 정한 조건에 따라 요약 또는 질의응답 작업을 선택할 수 있다.
- 허용되지 않은 중간 결과가 나오면 후속 작업을 실행하지 않고 원인을 기록할 수 있다.
- 분류 단계 실패와 후속 분석 실패를 구분하고, 원문·중간 결과·최종 결과를 함께 관리할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- CSV 3개를 노트북과 같은 폴더에 둡니다. 다른 위치라면 `DATA_DIR`를 수정합니다.
- 앞 강의의 노트북은 필요하지 않습니다. 이 파일을 위에서 아래로 실행합니다.

| 데이터 | 역할 |
|---|---|
| 엘지유플러스_분류.csv | 상담 요건 분류 지시·원문·기존 출력 확인 |
| 엘지유플러스_요약.csv | 같은 상담의 요약 지시와 기존 출력 확인 |
| 엘지유플러스_질의응답.csv | 같은 상담의 질문과 기존 답변 확인, 질의응답 경로의 입력 준비 |

공통 원본 컬럼은 `source_id`, `consulting_category`, `task_category`, `instruction`, `input`, `output`입니다. `source_id`는 상담 ID, `instruction`은 작업 지시·질문, `input`은 상담 원문, `output`은 CSV에 이미 저장된 출력입니다. 하나의 상담에 여러 작업이 있어 source_id가 반복됩니다.

### 실습 범위와 자료 구분

교안의 명언·리뷰 사례를 **상담 데이터**에 적용합니다. 기초~중급 수준의 필수 2문제와 같은 수준의 과제 1문제로 구성했습니다. 기본 문법과 제공 함수를 이용해 처리 흐름을 확인합니다.

이번 노트북은 **실제 LLM API를 호출하지 않습니다.** 모델이 응답했다고 가정한 값을 제공하고, 그 값을 순서대로 전달하거나 조건에 따라 다음 작업을 선택하는 프로그램을 작성합니다. API 키·유료 계정 없이 실행할 수 있습니다.

- CSV의 원문·지시·기존 출력은 실제 데이터입니다. 기존 output을 자동으로 새 모델 응답으로 사용하지 않습니다.
- 가상 분류·요약·답변·요청 상태는 작성자가 만든 학습 자료입니다. 실제 모델의 성능이나 오류 발생 빈도를 측정한 결과가 아닙니다.
- '가상 후속 단계 실행'은 준비된 응답을 선택해 흐름을 확인한다는 뜻입니다. 파이썬이 상담을 이해해 새 답변을 생성하는 것은 아닙니다.
- 요청 설계용 딕셔너리는 단계별 전달 정보를 표현합니다. 특정 API의 실제 요청 규격은 아닙니다.
- 호출 횟수 비교는 해당 설계를 실제 API로 구현할 때의 **정상 경로 기준 예상 횟수**입니다. 이 노트북의 실제 API 호출 횟수는 0회입니다.
- RAG 검색, Tool Calling, Agent의 자율적 행동 선택, 상세 API 연결, 자동 재시도, 정확도 지표 계산은 포함하지 않습니다.
- 원문과 가림 표시(`<NAME>`, `<DATE>` 등)를 유지합니다. 기존 출력도 원문에 비추어 확인할 참고 자료입니다.

| 교안 내용 | 대응 실습 |
|---|---|
| 단일 호출·순차 처리·조건부 분기 | 필수 1의 방식 비교, 필수 2의 경로 선택 |
| 분석 목적에 따른 워크플로우 선택 | 각 문제의 작업 의존성·필요 결과·시간과 비용 해석 |
| 단계별 입력·출력 연결과 처리 결과 확인 | 원문·분류 결과 전달, 과제 1의 실패 단계 구분 |


## 실습 준비

아래 코드는 제공 코드입니다. 실행 후 결과를 확인하세요. 함수 구현은 평가하지 않습니다.

1. 데이터 불러오기: 세 CSV를 별도 DataFrame으로 읽습니다.
2. 분석 대상 컬럼 확인: 원문·지시·기존 출력과 ID를 확인합니다.
3. 결측치 및 자료형 확인: 결측치·빈 문자열, 원본 컬럼의 자료형을 확인합니다.
4. 기초 통계량과 분포 확인: 고유 상담 수, 작업 유형별 행 수, 본문 길이를 확인합니다.

`record_id`는 파일 이름과 0부터 시작하는 행 위치를 결합한 실습용 행 식별자입니다. 원본의 영구 ID는 아니며 파일 행 순서가 바뀌면 달라집니다. 본문 글자 수는 토큰 수와 다릅니다.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])


### 워크플로우 입력 준비 — 제공 코드

상담 40001·40003·40005·40006·40007에서 **상담 요건 분류 한 행, 길이 제한 요약 한 행, 지정 질문 한 행**을 선택합니다. 세 행의 원문이 같은지 확인한 뒤 상담별 한 행의 `documents`로 정리합니다.

- `기존_category`, `기존_summary`, `기존_answer`는 CSV 참고 출력입니다.
- `question`은 이번 실습에서 각 상담에 미리 지정한 질문입니다. 모델이 자율적으로 질문을 고르는 구조가 아닙니다.
- 이번에는 상담마다 워크플로우를 한 번 적용하므로 source_id를 document_id로 사용합니다. 한 상담에 여러 질문을 각각 처리한다면 별도 작업 ID가 필요합니다.
- `get_document()`는 문서 한 행을 선택합니다.
- `make_follow_input()`은 원문·분류 결과·작업 지시를 후속 입력으로 묶기만 합니다. API 요청이나 응답 생성은 하지 않습니다.

요약 경로에는 모든 상담에 같은 요약 기준을 사용하고, 질의응답 경로에는 미리 준비한 질문을 함께 전달합니다. 원문은 분석의 근거, 분류 결과는 처리 방향을 정하는 정보입니다.


In [ ]:
question_specs = {
    40001: ("의문사형", "고객이 삼성 케어 플러스 보험 상품 가입 시 매장 코드를 어떻게 확인할 수 있나요?"),
    40003: ("의문사형", "고객의 결합 할인은 누구 앞으로 되어 있나요?"),
    40005: ("예/아니요형", "고객은 지금 추가 결합 할인을 받겠다고 했어?"),
    40006: ("예/아니요형", "고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?"),
    40007: ("예/아니요형", "상담사는 고객에게 가까운 대리점을 문자 메시지로 안내했습니까?"),
}
document_rows = []
for sid, (qa_type, question) in question_specs.items():
    cr = get_record(classification, sid, "상담 요건")
    sr = get_record(summarization, sid, "길이 제한 요약")
    qr = get_record(qa, sid, qa_type, question)
    if not (cr["input"] == sr["input"] == qr["input"]):
        raise ValueError("선택한 세 작업의 원문이 다릅니다. 원본부터 확인하세요.")
    document_rows.append({
        "document_id": sid, "source_id": sid, "text": cr["input"],
        "class_record_id": cr["record_id"], "summary_record_id": sr["record_id"],
        "qa_record_id": qr["record_id"], "question": qr["instruction"],
        "분류_기존지시": cr["instruction"], "요약_기존지시": sr["instruction"],
        "기존_category": cr["output"], "기존_summary": sr["output"], "기존_answer": qr["output"]
    })
documents = pd.DataFrame(document_rows)
print("상담별 입력 행 수:", len(documents))
print("문서 ID 고유:", documents["document_id"].is_unique)
print(documents[["document_id", "기존_category", "question"]].to_string(index=False))

ALLOWED_CATEGORIES = ["단일 요건 민원", "다수 요건 민원"]
CLASS_INSTRUCTION = (
    "상담 요건을 단일 요건 민원 또는 다수 요건 민원 중 하나로 분류하세요. "
    "중심 요청·주제가 하나면 단일, 구분되는 요청·주제가 둘 이상이면 다수입니다. "
    "한 요청의 절차를 여러 번 질문한 것만으로 다수로 보지 마세요. 범주명만 출력하세요."
)
SUMMARY_INSTRUCTION = (
    "제공된 상담 요건 분류를 참고하여 원문의 핵심 문의와 처리 상황을 요약하세요. "
    "여러 주제가 있으면 주제를 구분해 반영하고, 실제 완료와 추후 예정 사항을 구분하세요. "
    "원문에 없는 사실은 추가하지 마세요."
)

def get_document(sid):
    selected = documents[documents["document_id"] == sid]
    if len(selected) != 1:
        raise ValueError("문서 ID로 한 행을 선택해야 합니다.")
    return selected.iloc[0]

def make_follow_input(doc, category, route):
    if route == "상담 요약":
        instruction = SUMMARY_INSTRUCTION
    elif route == "질의응답":
        instruction = "상담 원문을 근거로 다음 질문에 답하세요. 질문: " + doc["question"]
    else:
        raise ValueError("정의되지 않은 후속 작업입니다.")
    return {"document_id": int(doc["document_id"]), "text": doc["text"],
            "category": category, "instruction": instruction}


---
## 필수 1. 단일 호출과 순차 처리 비교

### 문제 1-1. 분류·요약을 한 번에 얻는 방식과 단계로 나누는 방식

#### 문제 설명

상담 **40005**의 요건을 분류하고 핵심 내용을 요약하려고 합니다. 아래 두 설계를 비교합니다.

| 설계 | 처리 흐름 | 가정 |
|---|---|---|
| A: 단일 호출 | 원문을 넣어 category·summary를 함께 받음 | 두 결과를 별도 중간 처리 없이 한 번에 생성할 수 있음 |
| B: 순차 처리 | 원문으로 category 생성 → 원문+category로 summary 생성 → 결과 구성 | 분류 중간 결과를 확인하고 후속 요약에서 활용해야 함 |

제공 셀의 두 응답은 같은 내용을 담도록 작성한 가상 값입니다. 어느 방식의 실제 품질이 더 좋은지 측정하는 실험은 아닙니다.

#### 요구사항

1. `doc = get_document(40005)`로 문서를 선택하고 원문, 기존 분류·요약 지시와 출력을 확인하세요.
2. 단일 호출의 입력을 `single_input`에 구성하세요. document_id·text·instruction을 넣고, 지시는 분류와 요약을 함께 요구하도록 작성합니다. 기존 output을 입력에 넣지 않습니다.
3. 순차 처리의 분류 입력 `class_input`을 작성하고, 제공된 가상 `class_demo`가 허용 범주인지 확인하세요.
4. 허용되는 경우 `make_follow_input(doc, class_demo, "상담 요약")`으로 `summary_input`을 구성하세요. 원문과 분류 결과가 모두 전달되었는지 확인합니다.
5. 순차 처리의 단계별 `단계`, `입력 정보`, `출력 정보`, `LLM 요청 단계`를 담은 `stage_plan` 표를 작성하세요. 결과를 표에 모으는 단계도 표시하되 LLM 호출로 세지 않습니다.
6. `방식`, `document_id`, `원문`, `기존_category`, `기존_summary`, `새_category`, `새_summary`, `정상경로_예상호출수`를 포함한 `comparison` 표를 만드세요. A·B를 각각 한 행으로 작성합니다.

#### 해석 질문

**Q1.** 분류와 요약 두 결과가 필요하면 LLM을 반드시 두 번 호출해야 하나요? A를 우선 고려할 수 있는 조건을 설명하세요.

**Q2.** B의 작업 의존성은 어디에 있나요? 분류와 요약 코드를 단순히 앞뒤로 실행하는 것만으로 앞 결과를 활용한다고 할 수 있나요?

**Q3.** 요약 입력에서 원문을 빼고 분류명만 보내면 어떤 문제가 생기나요? 이번 원문에서 요약해야 할 실제 사실을 두 가지 적으세요.

**Q4.** B의 장점과 시간·비용·오류 전파 측면의 고려사항은 무엇인가요? 제공한 두 결과가 같다는 이유로 실제 품질이 같다고 결론 내릴 수 있나요?

#### 제출 결과

- 단일·분류·요약 단계의 입력 구성 코드와 전달 확인 결과
- stage_plan, comparison
- Q1~Q4 답변


### 제공 자료

아래 셀은 수정하지 않고 실행하세요.


In [ ]:
# 제공 자료: 실제 LLM 응답이 아닌 가상 값입니다.
demo_summary_40005 = (
    "통화 중 대기 서비스를 재설정하고 시험 통화로 정상 작동을 확인했다. "
    "추가 결합 할인은 즉시 연결하지 않고 추후 상담 연락을 받기로 했다."
)
single_demo = {"category": "다수 요건 민원", "summary": demo_summary_40005}
class_demo = "다수 요건 민원"
summary_demo = demo_summary_40005


In [ ]:
# 문제 1 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 필수 2. 조건부 분기와 후속 입력 연결

### 문제 2-1. 상담 요건에 따라 요약 또는 질의응답 선택하기

#### 문제 설명

이번 분석기의 목표는 **여러 주제가 있는 상담은 전체 문의와 처리 상황을 요약하고, 하나의 중심 요청을 가진 상담은 미리 정한 확인 질문에 답하는 것**입니다. 다음 경로를 개발자가 미리 정했다고 가정합니다.

| 분류 결과 | 선택할 경로 | 후속 입력 |
|---|---|---|
| 다수 요건 민원 | 상담 요약 | 원문 + 분류 결과 + 요약 지시 |
| 단일 요건 민원 | 질의응답 | 원문 + 분류 결과 + 지정 질문을 포함한 지시 |
| 그 외 값·결과 없음 | 경로 선택 안 함 | 오류를 확인한 뒤 후속 처리 여부 결정 |

이것은 **이번 분석 목적에 맞춘 연습 규칙**입니다. 모든 단일 상담은 질의응답만 해야 한다거나 다수 상담에는 질의응답을 할 수 없다는 뜻이 아닙니다.

상담 40005·40006에 대해 가상 분류와 후속 결과가 제공됩니다. 40006의 지정 질문은 `고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?`입니다.

#### 요구사항

1. `get_document(40006)`으로 원문과 지정 질문을 확인하세요. 40005 원문은 필수 1에서 확인한 내용을 사용합니다.
2. 위 조건에 따라 `"상담 요약"`, `"질의응답"`, `None` 중 하나를 반환하는 `choose_route(category)` 함수를 if·elif·else로 작성하세요.
3. 두 상담을 차례로 처리하면서 **제공된 가상 분류 결과**로 경로를 선택하세요. CSV의 기존_category를 분기 입력 대신 사용하지 않습니다.
4. 선택된 경로에 대해 `make_follow_input()`으로 후속 입력을 만들고 `follow_inputs` 목록에 보관하세요. ID·원문·분류 결과가 원래 값과 일치하는지 확인하고 지시를 출력합니다.
5. 선택된 경로의 가상 출력만 가져와 `branch_results`를 만드세요. document_id·text·기존 출력 3개·새 category·route·final_result를 보존합니다.
6. 별도 점검으로 `choose_route("복합 문의")`, `choose_route(None)`을 실행하세요. 이어서 40005가 잘못 `단일 요건 민원`으로 분류되었다고 가정할 때 선택될 경로를 확인하고, 원문에 맞는 분류인지 별도로 설명하세요. 이 점검 값으로 정상 결과를 덮어쓰지 않습니다.

#### 해석 질문

**Q1.** 각 상담은 어떤 경로로 가며 다음 단계에 어떤 정보가 필요한가요? 선택되지 않은 경로까지 모두 실행해야 하나요?

**Q2.** 분류·후속 분석 순서가 있다는 점에서 순차 처리와 비슷하지만, 이 설계에서 조건부 분기라고 부르는 이유는 무엇인가요?

**Q3.** `복합 문의`와 `단일 요건 민원`으로 잘못 분류된 40005는 무엇이 다른가요? 형식 오류와 오류 전파를 구분하여 설명하세요.

**Q4.** 이 워크플로우를 Agent의 자율적 작업 선택이라고 볼 수 있나요? LLM 분류 결과 외에 무엇을 기준으로 분기할 수 있나요?

#### 제출 결과

- choose_route 함수, 두 상담의 후속 입력과 전달 확인
- branch_results, 별도 경로 점검 결과
- Q1~Q4 답변


### 제공 자료

아래 셀은 수정하지 않고 실행하세요.


In [ ]:
# 제공 자료: 두 상담의 가상 중간 결과와 가상 후속 응답입니다.
branch_class_demo = {40005: "다수 요건 민원", 40006: "단일 요건 민원"}
branch_follow_demo = {
    (40005, "상담 요약"): "통화 중 대기 재설정 후 정상 작동을 확인했다. 추가 결합 할인은 추후 상담 연락을 받기로 했다.",
    (40006, "질의응답"): "네. 상담사는 매장을 방문해 해지할 때 명의자 신분증을 지참하라고 안내했다."
}


In [ ]:
# 문제 2 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 과제 1. 단계별 검증과 실패 처리

### 문제 3-1. 다섯 상담의 워크플로우 상태 관리

#### 문제 설명

필수 2의 분기 규칙을 다섯 상담에 적용합니다. 아래 가상 응답 기록을 재현하면서, 어디에서 문제가 생겼는지 남기세요. 필수 2의 `choose_route()`를 먼저 완성하고 사용합니다.

| 문서 | 분류 요청 상태 | 가상 분류 출력 | 후속 상황 |
|---|---|---|---|
| 40001 | completed | 다수 요건 민원 | 요약 응답 정상 |
| 40003 | completed | 다수 요건 민원 | 요약 요청 실패 |
| 40005 | completed | 복합 문의 | 분류 검증에서 중단해야 함 |
| 40006 | completed | 단일 요건 민원 | 질의응답 응답 정상 |
| 40007 | failed | 없음 | 분류 요청 실패로 중단해야 함 |

completed·failed는 실습용 요청 상태 값입니다. 분류 요청이 completed라고 해도 출력 값이 허용 범주가 아니면 후속 분석으로 진행할 수 없습니다.

상태를 다음과 같이 구분합니다.

| 기록 | 값 | 뜻 |
|---|---|---|
| class_status | success / error / not_run | 분류 단계의 검증 통과 / 오류 / 미실행 |
| follow_status | success / error / not_run | 후속 단계의 검증 통과 / 오류 / 미실행 |
| status | success / error | 전체 워크플로우의 정상 완료 여부 |
| failed_stage | 입력 준비 / 분류 / 분기 / 후속 분석 / 빈 문자열 | 문제가 발견된 단계 |
| error_reason | 설명 문자열 | 실패하거나 중단한 원인 |

이 예시의 success는 처리 및 기본 출력 조건을 통과했다는 뜻입니다. 원문과의 의미 일치까지 자동 검증했다는 뜻이 아닙니다. 기록된 실패 원인은 실제 서비스에서 발생한 장애가 아닙니다.

#### 요구사항

1. documents의 다섯 행을 순회하며 결과 행을 하나씩 만드세요. 시작할 때 category·route·final_result는 None, 단계 상태는 not_run, 전체 status는 error로 둡니다.
2. 원문이 없거나 비어 있으면 `입력 준비` 오류로 기록하고 다음 문서로 넘어갑니다. 제공된 실제 다섯 원문은 모두 존재하므로 정상 경로에서는 이 오류가 발생하지 않습니다.
3. `class_events`에서 해당 문서의 분류 기록을 확인합니다. 요청 실패, 출력 없음, 허용 범주가 아닌 값은 `분류` 오류로 기록하고 후속 응답을 조회하지 마세요. `raw_category`에는 제공된 원래 출력을 남깁니다.
4. 유효한 분류이면 category에 저장하고 class_status를 success로 설정합니다. choose_route()로 경로를 고르며, 경로가 없으면 `분기` 오류로 기록합니다.
5. 경로가 있으면 make_follow_input()으로 후속 입력을 만들고 `workflow_inputs`에 보관하세요. 이어서 해당 문서 ID를 `follow_attempted_ids`에 기록하고 `follow_events`의 가상 후속 응답을 읽습니다.
6. 후속 요청이 실패했거나 결과가 없으면 `후속 분석` 오류로 기록하고 final_result는 None으로 유지하세요. 정상인 경우에만 follow_status·status를 success로 바꾸고 최종 결과를 저장합니다. 분류가 성공한 뒤 후속 단계가 실패해도 이미 얻은 category는 보존합니다.
7. `workflow_results`에 입력 문서 5개를 모두 한 행씩 남기세요. document_id·원문·기존 출력 외에 raw_category·category·route·class_status·follow_status·final_result·status·failed_stage·error_reason을 포함합니다.
8. 전체 성공 건수와 실패 단계별 건수를 집계하고, `review_needed`를 만드세요. `follow_attempted_ids`에 40005·40007이 들어가지 않았는지도 확인하세요. 실제 재요청은 수행하지 않습니다.

**보조 구문:** `for _, doc in documents.iterrows():`, `row = doc.to_dict()`, `row.update({...})`, `rows.append(row)`, `continue`. `continue`는 현재 문서의 처리를 끝내고 다음 문서로 넘어가는 문법입니다. 중단하기 전에 결과 행을 추가해야 해당 문서의 기록이 남습니다.

#### 해석 질문

**Q1.** 정상 완료 문서는 몇 개이며 어떤 ID인가요? 분류 단계 오류와 후속 분석 오류의 ID도 구분하세요.

**Q2.** 40005와 40007은 왜 후속 분석을 실행하지 않나요? 두 문서 모두 최종 결과가 없지만 원인은 어떻게 다른가요?

**Q3.** 40003을 다시 처리한다면 분류부터 반드시 다시 해야 하나요? 유지해야 할 정보와 재처리 전 확인할 사항을 적으세요.

**Q4.** 최종 결과가 있는 행만 남기거나, 실패 행의 category·final_result를 기존 출력으로 채우면 어떤 문제가 있나요? 전체 결과 연결 검증에는 무엇이 필요한가요?

#### 제출 결과

- 입력·중간 결과 검증 및 중단 코드
- workflow_results 5행, workflow_inputs, follow_attempted_ids
- 성공·실패 집계, review_needed, Q1~Q4 답변


### 제공 자료

아래 셀은 수정하지 않고 실행하세요.


In [ ]:
# 제공 자료: 가상 응답 기록. 실제 API 요청을 실행하지 않습니다.
class_events = {
    40001: {"request_status": "completed", "output": "다수 요건 민원", "error": ""},
    40003: {"request_status": "completed", "output": "다수 요건 민원", "error": ""},
    40005: {"request_status": "completed", "output": "복합 문의", "error": ""},
    40006: {"request_status": "completed", "output": "단일 요건 민원", "error": ""},
    40007: {"request_status": "failed", "output": None, "error": "분류 요청 실패(가상)"},
}
follow_events = {
    40001: {"request_status": "completed", "output": "고객은 사은품 케이스, 보험 매장 코드, 카드 할인과 휴대폰 정지 횟수를 문의하고 관련 안내를 받았다.", "error": ""},
    40003: {"request_status": "failed", "output": None, "error": "요약 요청 실패(가상)"},
    40006: {"request_status": "completed", "output": "네. 상담사는 매장에서 해지할 때 명의자 신분증을 지참하라고 안내했다.", "error": ""},
}
# 40005·40007의 후속 응답은 없습니다. 앞 단계에서 중단해야 하는 사례입니다.
print("분류 기록 ID:", list(class_events))
print("후속 응답 기록 ID:", list(follow_events))


In [ ]:
# 문제 3 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 실습 마무리

다음 질문에 각각 2~3문장으로 답하세요.

1. **어떤 분석 문제가 있었는가?** 작업 의존성, 조건별 경로, 중간 결과 오류와 단계별 실패를 정리하세요.
2. **어떤 방법을 적용했는가?** 단일 호출·순차 처리·조건부 분기의 선택과 입력 전달, 검증·중단·기록 방법을 설명하세요.
3. **무엇을 근거로 결론을 내렸는가?** 전달된 원문과 분류 값, 선택 경로, 단계별 상태, 최종 결과와 ID를 구분해 설명하세요.

### 제출 전 확인

- 필수 1·필수 2·과제 1의 요구사항과 Q1~Q4를 모두 작성했나요?
- 앞 단계 출력이 실제로 후속 입력에 들어가고, 원문도 함께 전달되나요?
- 미리 정한 규칙에 맞는 경로를 선택했나요?
- 허용되지 않은 분류나 분류 실패에서 후속 단계를 중단했나요?
- 실패와 미실행을 구분하고 원문·중간 결과를 보존했나요?
- 가상 응답 기반 결과를 실제 모델의 품질·비용 측정 결과로 해석하지 않았나요?


### 마무리 답안

1.
2.
3.
